In [0]:
# =========================================================
# 00_LANDING_ORDERS
# =========================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType
)
from functools import reduce
import uuid


# =========================================================
# 1. CONFIG
# =========================================================

source_directory = "/Volumes/workspace/default/landing/"
pipeline_name = "orders_pipeline"
lookback_days = 3

processed_files_table = "workspace.control.processed_files"
watermark_table = "workspace.control.watermark"
audit_table = "workspace.control.etl_run_log"
landing_table = "workspace.landing.orders"
quarantine_table = "workspace.control.orders_ingestion_quarantine"


# =========================================================
# 2. HELPER - WRITE PROCESSED FILE METADATA
# =========================================================

def mark_files_processed(
    files,
    run_id,
    file_row_counts,
    rows_landed
):

    rows = [
        (
            f["file_path"],
            f["file_name"],
            f["file_size"],
            str(f["modification_time"]),
            pipeline_name,
            run_id,
            file_row_counts.get(f["file_path"], 0)
        )
        for f in files
    ]

    schema = StructType([
        StructField("file_path", StringType(), False),
        StructField("file_name", StringType(), False),
        StructField("file_size", LongType(), False),
        StructField("modification_time_raw", StringType(), False),
        StructField("pipeline_name", StringType(), False),
        StructField("run_id", StringType(), False),
        StructField("rows_read", LongType(), True)
    ])

    df = (
        spark.createDataFrame(rows, schema=schema)
        .withColumn(
            "modification_time",
            F.to_timestamp("modification_time_raw")
        )
        .drop("modification_time_raw")
        .withColumn(
            "processed_at",
            F.current_timestamp()
        )
        .withColumn(
            "status",
            F.lit("PROCESSED")
        )
        .withColumn(
            "rows_landed",
            F.lit(rows_landed).cast("bigint")
        )
        .select(
            "file_path",
            "file_name",
            "file_size",
            "modification_time",
            "pipeline_name",
            "processed_at",
            "run_id",
            "status",
            "rows_read",
            "rows_landed"
        )
    )

    (
        df.write
        .mode("append")
        .saveAsTable(processed_files_table)
    )


# =========================================================
# 3. ACTIVE RUN GUARDRAIL
# =========================================================

active_runs = spark.sql(f"""
    SELECT run_id, batch_id
    FROM {audit_table}
    WHERE pipeline_name = '{pipeline_name}'
      AND status = 'RUNNING'
""").collect()

if len(active_runs) > 0:
    raise ValueError(
        f"Landing blocked: {len(active_runs)} RUNNING "
        f"pipeline run(s) already exist."
    )


# =========================================================
# 4. LIST SOURCE FILES
# =========================================================

all_files = [
    f
    for f in dbutils.fs.ls(source_directory)
    if f.name.startswith("orders_batch_")
    and f.name.endswith(".csv")
]

if not all_files:
    dbutils.notebook.exit("NO_SOURCE_FILES")


file_rows = [
    (
        f.path,
        f.name,
        f.size,
        f.modificationTime
    )
    for f in all_files
]

file_schema = StructType([
    StructField("file_path", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("file_size", LongType(), False),
    StructField("modification_time_ms", LongType(), False)
])

df_files = (
    spark.createDataFrame(file_rows, schema=file_schema)
    .withColumn(
        "modification_time",
        (F.col("modification_time_ms") / 1000).cast("timestamp")
    )
    .drop("modification_time_ms")
)


# =========================================================
# 5. ONE-TIME BOOTSTRAP
# =========================================================

processed_count = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM {processed_files_table}
    WHERE pipeline_name = '{pipeline_name}'
""").first()["cnt"]

if processed_count == 0:

    bootstrap_df = (
        df_files
        .withColumn("pipeline_name", F.lit(pipeline_name))
        .withColumn("processed_at", F.current_timestamp())
        .withColumn("run_id", F.lit("INITIAL_BOOTSTRAP"))
        .withColumn("status", F.lit("PROCESSED"))
        .withColumn("rows_read", F.lit(None).cast("bigint"))
        .withColumn("rows_landed", F.lit(None).cast("bigint"))
        .select(
            "file_path",
            "file_name",
            "file_size",
            "modification_time",
            "pipeline_name",
            "processed_at",
            "run_id",
            "status",
            "rows_read",
            "rows_landed"
        )
    )

    bootstrap_files = bootstrap_df.count()

    (
        bootstrap_df.write
        .mode("append")
        .saveAsTable(processed_files_table)
    )

    print(f"BOOTSTRAP COMPLETE | files={bootstrap_files}")

    dbutils.notebook.exit("BOOTSTRAP_COMPLETE")


# =========================================================
# 6. FIND NEW / CHANGED FILES
#
# File version:
# path + size + modification_time
# =========================================================

df_processed = (
    spark.table(processed_files_table)
    .filter(
        (F.col("pipeline_name") == pipeline_name)
        & (F.col("status") == "PROCESSED")
    )
    .select(
        "file_path",
        "file_size",
        "modification_time"
    )
)

df_new_files = (
    df_files.alias("f")
    .join(
        df_processed.alias("p"),
        (
            (F.col("f.file_path") == F.col("p.file_path"))
            & (F.col("f.file_size") == F.col("p.file_size"))
            & (
                F.col("f.modification_time")
                == F.col("p.modification_time")
            )
        ),
        "left_anti"
    )
)

new_files = df_new_files.collect()

print(
    f"FILES | total={len(all_files)} "
    f"| new/changed={len(new_files)}"
)

if not new_files:
    print("NO NEW FILES | nothing to process")
    dbutils.notebook.exit("NO_NEW_FILES")

for f in new_files:
    print(f"NEW FILE | {f['file_name']}")


# =========================================================
# 7. WATERMARK
# =========================================================

watermark_rows = spark.sql(f"""
    SELECT watermark_timestamp
    FROM {watermark_table}
    WHERE pipeline_name = '{pipeline_name}'
""").collect()

if len(watermark_rows) != 1:
    raise ValueError(
        f"Expected exactly 1 watermark row, "
        f"found {len(watermark_rows)}"
    )

current_watermark = watermark_rows[0]["watermark_timestamp"]

effective_watermark = spark.sql(f"""
    SELECT TIMESTAMPADD(
        DAY,
        -{lookback_days},
        TIMESTAMP '{current_watermark}'
    ) AS ts
""").first()["ts"]


# =========================================================
# 8. RUN SETUP
# =========================================================

run_id = str(uuid.uuid4())
audit_created = False
df_new_rows = None


try:

    # =====================================================
    # 9. SOURCE CONTRACTS
    # =====================================================

    schema_v1 = {
        "order_id",
        "customer_id",
        "amount",
        "status",
        "order_date",
        "last_updated",
        "event_id",
        "batch_id",
        "source_file"
    }

    schema_v2 = {
        "order_id",
        "customer_id",
        "order_amount",
        "currency",
        "status",
        "order_date",
        "last_updated",
        "event_id",
        "batch_id",
        "source_file"
    }

    schema_v3 = {
        "order_id",
        "customer_id",
        "amount",
        "currency",
        "status",
        "order_date",
        "last_updated",
        "event_id",
        "batch_id",
        "source_file"
    }

        # =====================================================
    # 9. SOURCE CONTRACTS
    # =====================================================

    schema_v1 = {
        "order_id",
        "customer_id",
        "amount",
        "status",
        "order_date",
        "last_updated",
        "event_id",
        "batch_id",
        "source_file"
    }

    schema_v2 = {
        "order_id",
        "customer_id",
        "order_amount",
        "currency",
        "status",
        "order_date",
        "last_updated",
        "event_id",
        "batch_id",
        "source_file"
    }

    schema_v3 = {
        "order_id",
        "customer_id",
        "amount",
        "currency",
        "status",
        "order_date",
        "last_updated",
        "event_id",
        "batch_id",
        "source_file"
    }

    schema_v4 = {
        "order_id",
        "customer_id",
        "amount",
        "currency",
        "payment_method",
        "status",
        "order_date",
        "last_updated",
        "event_id",
        "batch_id",
        "source_file"
    }


    # =====================================================
    # 10. READ ONLY NEW / CHANGED FILES
    # =====================================================

    normalized_dfs = []
    file_row_counts = {}

    for file_meta in new_files:

        file_path = file_meta["file_path"]
        file_name = file_meta["file_name"]

        df_file = (
            spark.read
            .option("header", True)
            .option("inferSchema", False)
            .csv(file_path)
        )

        source_columns = set(df_file.columns)

        if source_columns == schema_v1:

            schema_version = "v1"

            df_normalized = (
                df_file
                .withColumn(
                    "currency",
                    F.lit(None).cast("string")
                )
                .withColumn(
                    "payment_method",
                    F.lit(None).cast("string")
                )
            )

        elif source_columns == schema_v2:

            schema_version = "v2"

            df_normalized = (
                df_file
                .withColumnRenamed("order_amount", "amount")
                .withColumn(
                    "payment_method",
                    F.lit(None).cast("string")
                )
            )

        elif source_columns == schema_v3:

            schema_version = "v3"

            df_normalized = (
                df_file
                .withColumn(
                    "payment_method",
                    F.lit(None).cast("string")
                )
            )

        elif source_columns == schema_v4:

            schema_version = "v4"

            df_normalized = df_file

        else:

            raise ValueError(
                "SOURCE SCHEMA CONTRACT VIOLATION | "
                f"file={file_name} | "
                f"columns={sorted(source_columns)}"
            )

        rows_in_file = df_normalized.count()

        file_row_counts[file_path] = rows_in_file
        normalized_dfs.append(df_normalized)

        print(
            f"READ | {file_name} "
            f"| schema={schema_version} "
            f"| rows={rows_in_file}"
        )


    # =====================================================
    # 11. UNION + SAFE TIMESTAMP PARSE
    # =====================================================

    df_source = reduce(
        lambda a, b: a.unionByName(b),
        normalized_dfs
    )

    total_rows_read = df_source.count()

    df_source = df_source.withColumn(
        "_last_updated_ts",
        F.expr("try_to_timestamp(last_updated)")
    )


    # =====================================================
    # 12. INGESTION QUARANTINE
    # =====================================================

    df_invalid_ts = df_source.filter(
        F.col("last_updated").isNotNull()
        & F.col("_last_updated_ts").isNull()
    )

    invalid_ts_rows = df_invalid_ts.count()

    df_existing_quarantine = (
        spark.table(quarantine_table)
        .select(
            "batch_id",
            "event_id",
            "last_updated"
        )
    )

    df_quarantine_new = (
        df_invalid_ts.alias("s")
        .join(
            df_existing_quarantine.alias("q"),
            (
                F.col("s.batch_id").eqNullSafe(F.col("q.batch_id"))
                & F.col("s.event_id").eqNullSafe(F.col("q.event_id"))
                & F.col("s.last_updated").eqNullSafe(
                    F.col("q.last_updated")
                )
            ),
            "left_anti"
        )
        .withColumn(
            "quarantine_reason",
            F.lit("INVALID_LAST_UPDATED")
        )
        .withColumn(
            "quarantined_at",
            F.current_timestamp()
        )
        .withColumn(
            "run_id",
            F.lit(run_id)
        )
        .select(
            "order_id",
            "customer_id",
            "amount",
            "status",
            "order_date",
            "last_updated",
            "event_id",
            "batch_id",
            "source_file",
            "quarantine_reason",
            "quarantined_at",
            "run_id"
        )
    )

    quarantine_new_rows = df_quarantine_new.count()

    if quarantine_new_rows > 0:
        (
            df_quarantine_new.write
            .mode("append")
            .saveAsTable(quarantine_table)
        )


    # =====================================================
    # 13. LOOKBACK
    # =====================================================

    df_lookback = (
        df_source
        .filter(F.col("_last_updated_ts").isNotNull())
        .filter(
            F.col("_last_updated_ts")
            > F.lit(effective_watermark)
        )
    )

    lookback_rows = df_lookback.count()


    # =====================================================
    # 14. LANDING IDEMPOTENCY
    #
    # batch_id + event_id + last_updated
    # =====================================================

    df_existing_landing = (
        spark.table(landing_table)
        .select(
            "batch_id",
            "event_id",
            "last_updated"
        )
    )

    df_new_rows = (
    df_lookback.alias("s")
    .join(
        df_existing_landing.alias("l"),
        (
            F.col("s.batch_id").eqNullSafe(F.col("l.batch_id"))
            & F.col("s.event_id").eqNullSafe(F.col("l.event_id"))
            & F.col("s.last_updated").eqNullSafe(
                F.col("l.last_updated")
            )
        ),
        "left_anti"
    )
    .dropDuplicates(
        [
            "batch_id",
            "event_id",
            "last_updated"
        ]
    )
)

    # IMPORTANT:
    # Materialize all required run-level values BEFORE Landing changes.

    landing_metrics = (
        df_new_rows
        .agg(
            F.count("*").alias("landing_rows"),
            F.max("_last_updated_ts").alias("max_ingested_ts")
        )
        .first()
    )

    landing_rows = landing_metrics["landing_rows"]
    max_ingested_ts = landing_metrics["max_ingested_ts"]


    # =====================================================
    # LATE ARRIVING ROWS
    # =====================================================

    late_arriving_rows_ingested = (
        df_new_rows
        .filter(
            F.col("_last_updated_ts") <= F.lit(current_watermark)
        )
        .count()
    )

    # =====================================================
    # 15. NO NEW LANDING DATA
    # =====================================================

    if landing_rows == 0:

        mark_files_processed(
            files=new_files,
            run_id=run_id,
            file_row_counts=file_row_counts,
            rows_landed=0
        )

        print(
            f"COMPLETE | rows_read={total_rows_read} "
            f"| invalid_ts={invalid_ts_rows} "
            f"| landing=0"
        )

        dbutils.notebook.exit(
            "FILES_PROCESSED_NO_NEW_LANDING_ROWS"
        )


    # =====================================================
    # 16. BATCH GUARDRAIL
    # =====================================================

    new_batches = [
        r["batch_id"]
        for r in (
            df_new_rows
            .select("batch_id")
            .distinct()
            .collect()
        )
    ]

    if len(new_batches) != 1:
        raise ValueError(
            f"Expected exactly 1 new batch_id, "
            f"found {new_batches}"
        )

    batch_id = new_batches[0]


    # =====================================================
    # 17. CREATE AUDIT RUN
    # =====================================================

    spark.sql(f"""
        INSERT INTO {audit_table} (
            run_id,
            batch_id,
            pipeline_name,
            start_timestamp,
            end_timestamp,
            landing_rows,
            bronze_rows,
            silver_rows,
            gold_inserted,
            gold_updated,
            rejected_rows,
            status,
            error_message
        )
        VALUES (
            '{run_id}',
            '{batch_id}',
            '{pipeline_name}',
            CURRENT_TIMESTAMP(),
            NULL,
            NULL,
            NULL,
            NULL,
            NULL,
            NULL,
            NULL,
            'RUNNING',
            NULL
        )
    """)

    audit_created = True


    # =====================================================
    # 18. WRITE LANDING
    # =====================================================

    df_landing_write = (
            df_new_rows
            .withColumn(
                "load_timestamp",
                F.current_timestamp()
            )
            .select(
                "order_id",
                "customer_id",
                "amount",
                "currency",
                "payment_method",
                "status",
                "order_date",
                "last_updated",
                "event_id",
                "batch_id",
                "source_file",
                "load_timestamp"
            )
        )

    (
        df_landing_write.write
        .mode("append")
        .saveAsTable(landing_table)
    )


    # =====================================================
    # 19. AUDIT LANDING
    # =====================================================

    spark.sql(f"""
    UPDATE {audit_table}

    SET
        landing_rows = {landing_rows},
        late_arriving_rows_ingested = {late_arriving_rows_ingested}

    WHERE run_id = '{run_id}'
      AND status = 'RUNNING'
""")


    # =====================================================
    # 20. ADVANCE WATERMARK
    #
    # Monotonic - never backwards.
    # =====================================================

    new_watermark = current_watermark

    if (
        max_ingested_ts is not None
        and max_ingested_ts > current_watermark
    ):

        spark.sql(f"""
            UPDATE {watermark_table}
            SET
                watermark_timestamp =
                    TIMESTAMP '{max_ingested_ts}',
                updated_at = CURRENT_TIMESTAMP()
            WHERE pipeline_name = '{pipeline_name}'
        """)

        new_watermark = max_ingested_ts


    # =====================================================
    # 21. MARK FILE VERSION AS PROCESSED
    # =====================================================

    mark_files_processed(
        files=new_files,
        run_id=run_id,
        file_row_counts=file_row_counts,
        rows_landed=landing_rows
    )


    # =====================================================
    # 22. SUMMARY
    # =====================================================

    print("==============================================")
    print("LANDING COMPLETE")
    print("==============================================")
    print(f"Run ID:          {run_id}")
    print(f"Batch:           {batch_id}")
    print(f"Files processed: {len(new_files)}")
    print(f"Rows read:       {total_rows_read}")
    print(f"Invalid TS:      {invalid_ts_rows}")
    print(f"New quarantine:  {quarantine_new_rows}")
    print(f"Lookback rows:   {lookback_rows}")
    print(f"Landing rows:    {landing_rows}")
    print(f"Late arrivals:   {late_arriving_rows_ingested}")
    print(f"Watermark:       {current_watermark} -> {new_watermark}")
    print("Status:          RUNNING")
    print("Next:            01_BRONZE_ORDERS")
    print("==============================================")


except Exception as e:

    # =====================================================
    # 23. FAILURE HANDLING
    # =====================================================

    error_message = str(e)

    print(f"LANDING FAILED | {error_message}")

    if audit_created:

        safe_error = (
            error_message
            .replace("'", "''")[:2000]
        )

        try:
            spark.sql(f"""
                UPDATE {audit_table}
                SET
                    end_timestamp = CURRENT_TIMESTAMP(),
                    status = 'FAILED',
                    error_message = '{safe_error}'
                WHERE run_id = '{run_id}'
                  AND status = 'RUNNING'
            """)

        except Exception as audit_error:
            print(
                f"WARNING | audit update failed: "
                f"{audit_error}"
            )

    raise

In [0]:
%sql
select * from workspace.control.watermark

In [0]:
%sql
select * from control.orders_ingestion_quarantine

In [0]:
%sql
select * from control.processed_files

In [0]:
%sql
select * from control.etl_run_log